# IAQP Reproduction 1: Training Checkpoints

This notebook is the clean entry point for training the checkpoints used in the paper. It follows the protocol described in `latex_results/tex/HamedB_VLDB_2026/sec/4_exp.tex`:

- frozen CLIP embeddings
- 3 training epochs
- batch size 2048
- training budgets `10..100`
- checkpoint directories under `notebooks/outputs/checkpoints/`

The commands below are intentionally explicit. Replace the dataset cache paths with the local cache files available on your machine.

In [ ]:
from pathlib import Path

REPO_ROOT = Path('..').resolve()
OUTPUT_ROOT = REPO_ROOT / 'notebooks' / 'outputs'
CHECKPOINT_ROOT = OUTPUT_ROOT / 'checkpoints'

REPO_ROOT, CHECKPOINT_ROOT

## Dataset cache paths

The paper uses:

- `LAION-10M`
- `T2I-10M`
- `DataComp-8.2M`

The commands below expect precomputed embedding caches such as:

- `/ssd/hamed/ann/laion/precompute/laion_cache_up_to_10.pkl`
- `/ssd/hamed/ann/t2i-10M/precompute/t2i_cache_up_to_0.pkl`
- `/ssd/hamed/ann/datacomp_small/precompute/datacomp_cache_up_to_8200000.pkl`


In [ ]:
dataset_paths = {
    'laion_10m': '/ssd/hamed/ann/laion/precompute/laion_cache_up_to_10.pkl',
    't2i_10m': '/ssd/hamed/ann/t2i-10M/precompute/t2i_cache_up_to_0.pkl',
    'datacomp_8_2m': '/ssd/hamed/ann/datacomp_small/precompute/datacomp_cache_up_to_8200000.pkl',
}
dataset_paths

## Optional shared PCA for cross-dataset transfer

Cross-dataset results in the paper use a shared PCA rotation for LAION and DataComp. If your caches do not already contain the shared PCA variants, precompute them first.

In [ ]:
%%bash
cd ..

# Example: fit PCA on a cache and save the transform.
# Repeat as needed for your shared-PCA preparation pipeline.
# python scripts/precompute_pca.py \
#   --cache_pkl /ssd/hamed/ann/laion/precompute/laion_cache_up_to_10.pkl \
#   --out_npz notebooks/outputs/pca/laion_10m_pca_200.npz \
#   --device cuda \
#   --center \
#   --d_keep 200


## Main training runs used in the paper

These are the canonical mixed-teacher runs. They produce the checkpoints later consumed by the evaluation notebooks.

In [ ]:
%%bash
cd ..

# LAION-10M, CAGRA-trained IAQP
CUDA_VISIBLE_DEVICES=0 python -m core.main \
  --dataset laion \
  --data_path /ssd/hamed/ann/laion/precompute/laion_cache_up_to_10.pkl \
  --backend cuvs_cagra \
  --epochs 3 \
  --batch_size 2048 \
  --save_path notebooks/outputs/checkpoints/laion-10m_cuvs_cagra_up_to_e3

# LAION-10M, IVF-trained IAQP
CUDA_VISIBLE_DEVICES=0 python -m core.main \
  --dataset laion \
  --data_path /ssd/hamed/ann/laion/precompute/laion_cache_up_to_10.pkl \
  --backend ivf \
  --epochs 3 \
  --batch_size 2048 \
  --save_path notebooks/outputs/checkpoints/laion-10m_ivf_up_to_e3


In [ ]:
%%bash
cd ..

# T2I-10M mixed-teacher runs
CUDA_VISIBLE_DEVICES=0 python -m core.main \
  --dataset t2i \
  --data_path /ssd/hamed/ann/t2i-10M/precompute/t2i_cache_up_to_0.pkl \
  --backend cuvs_cagra \
  --epochs 3 \
  --batch_size 2048 \
  --save_path notebooks/outputs/checkpoints/t2i-10m_cuvs_cagra_up_to_e3

CUDA_VISIBLE_DEVICES=0 python -m core.main \
  --dataset t2i \
  --data_path /ssd/hamed/ann/t2i-10M/precompute/t2i_cache_up_to_0.pkl \
  --backend ivf \
  --epochs 3 \
  --batch_size 2048 \
  --save_path notebooks/outputs/checkpoints/t2i-10m_ivf_up_to_e3


In [ ]:
%%bash
cd ..

# DataComp-8.2M mixed-teacher runs
CUDA_VISIBLE_DEVICES=0 python -m core.main \
  --dataset datacomp \
  --data_path /ssd/hamed/ann/datacomp_small/precompute/datacomp_cache_up_to_8200000.pkl \
  --backend cuvs_cagra \
  --epochs 3 \
  --batch_size 2048 \
  --save_path notebooks/outputs/checkpoints/datacomp-8.2m_cuvs_cagra_up_to_e3

CUDA_VISIBLE_DEVICES=0 python -m core.main \
  --dataset datacomp \
  --data_path /ssd/hamed/ann/datacomp_small/precompute/datacomp_cache_up_to_8200000.pkl \
  --backend ivf \
  --epochs 3 \
  --batch_size 2048 \
  --save_path notebooks/outputs/checkpoints/datacomp-8.2m_ivf_up_to_e3


## Ablation checkpoints

The paper also reports ANN-only and Exact-K-only teacher variants. Use the same command shape, changing only the teacher selection logic.

In [ ]:
%%bash
cd ..

# Exact-K-only teacher example
CUDA_VISIBLE_DEVICES=0 python -m core.main \
  --dataset laion \
  --data_path /ssd/hamed/ann/laion/precompute/laion_cache_up_to_10.pkl \
  --backend exact_k \
  --epochs 3 \
  --batch_size 2048 \
  --save_path notebooks/outputs/checkpoints/laion-10m_exact_k_up_to_e3

# ANN-only CAGRA teacher example
CUDA_VISIBLE_DEVICES=0 python -m core.main \
  --dataset laion \
  --data_path /ssd/hamed/ann/laion/precompute/laion_cache_up_to_10.pkl \
  --backend cuvs_cagra \
  --disable_exactK_teacher \
  --epochs 3 \
  --batch_size 2048 \
  --save_path notebooks/outputs/checkpoints/laion-10m_cuvs_cagra_only_up_to_e3

# ANN-only IVF teacher example
CUDA_VISIBLE_DEVICES=0 python -m core.main \
  --dataset laion \
  --data_path /ssd/hamed/ann/laion/precompute/laion_cache_up_to_10.pkl \
  --backend ivf \
  --disable_exactK_teacher \
  --epochs 3 \
  --batch_size 2048 \
  --save_path notebooks/outputs/checkpoints/laion-10m_ivf_only_up_to_e3


## Expected artifacts

Each run writes a checkpoint directory containing `epoch_1.pt`, `epoch_2.pt`, `epoch_3.pt`, and `best.pt`. The next notebook consumes those directories directly.